In [1]:
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split
from collections import OrderedDict
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import scanpy as sc
import numpy as np
import pandas as pd
from torchsummary import summary
from torchviz import make_dot
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
import csv
from scipy.sparse import issparse



from model.dataset import BagsDataset, custom_collate_fn
from  model.model import MIL, EarlyStopping
from model.dataset import BagsDataset, custom_collate_fn

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    gpu_name = torch.cuda.get_device_name(torch.cuda.current_device())
    print(f"Using device: {device} ({gpu_name})")
else:
    print(f"Using device: {device}")

Using device: cpu


In [5]:
all_genes = pd.read_csv('tumor_antigen.csv')
all_genes = all_genes['Gene'].values
dataset = BagsDataset('training_visium.csv', immune_cell='tcell')
#adata = sc.read_h5ad('../test.h5ad')
#dataset = BagsDataset(adata, immune_cell='tcell',radius=200, resolution='low')
train_size = int(0.7 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])


Processing: adata=UKF275.h5ad, radius=200, resolution=high
Processing: adata=TumE2.h5ad, radius=200, resolution=high
Processing: adata=TumA1.h5ad, radius=200, resolution=high
Processing: adata=p16.h5ad, radius=200, resolution=high
Processing: adata=T_cell.h5ad, radius=200, resolution=low
Processing: adata=T_cell.h5ad, radius=200, resolution=low
Processing: adata=T_cell.h5ad, radius=200, resolution=low
Processing: adata=T_cell.h5ad, radius=200, resolution=low
Processing: adata=T_cell.h5ad, radius=200, resolution=low
Processing: adata=T_cell.h5ad, radius=200, resolution=low
Processing: adata=T_cell.h5ad, radius=200, resolution=low
Processing: adata=T_cell.h5ad, radius=200, resolution=low
Processing: adata=T_cell.h5ad, radius=200, resolution=low
Processing: adata=T_cell.h5ad, radius=200, resolution=low
Processing: adata=T_cell.h5ad, radius=200, resolution=low
Processing: adata=T_cell.h5ad, radius=200, resolution=low
Processing: adata=T_cell.h5ad, radius=200, resolution=low
Processing: ada

Creating Bags with radius 200: 100%|█████████████████████████| 3460/3460 [00:00<00:00, 22535.07it/s]


Total bags created: 69234
Average instances per bag: 8


In [6]:
dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)



In [8]:
model = MIL(all_genes).to(device)
criterion = nn.BCELoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)


In [9]:
num_epochs = 1000
patience = 5
early_stopping = EarlyStopping(patience=patience, delta=0.0001)

In [10]:
ig_scores_before_training = torch.sigmoid(model.immunogenicity.ig)
with open('ig_scores_before_training.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Gene', 'IG Score Before Training'])
    for gene, score in zip(all_genes, ig_scores_before_training):
        writer.writerow([gene, score.item()])

In [1]:
for epoch in range(num_epochs):
    model.train() 
    running_loss = 0.0
    
    with tqdm(dataloader, unit="batch") as tepoch:
        for i, (distances, gene_expressions, label, core_idx, current_genes) in enumerate(tepoch):
            tepoch.set_description(f"Epoch {epoch+1}/{num_epochs}")

            optimizer.zero_grad()

            distances = torch.stack(distances).to(device)
            gene_expressions = torch.stack(gene_expressions).to(device)
            label = label.clone().detach().float().to(device)
            
            output = model(distances, gene_expressions, list(current_genes[0]))
            
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            tepoch.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(dataloader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')
    

    model.eval()
    val_loss = 0.0
    all_labels = []
    all_outputs = []
    with torch.no_grad():
        for val_distances, val_gene_expressions, val_label, _, val_current_genes in val_dataloader:
            val_distances = torch.stack(val_distances).to(device)
            val_gene_expressions = torch.stack(val_gene_expressions).to(device)
            val_label = val_label.clone().detach().float().to(device)
            val_output = model(val_distances, val_gene_expressions, list(val_current_genes[0]))
            val_loss += criterion(val_output, val_label).item()
            all_labels.extend(val_label.cpu().numpy())
            all_outputs.extend(val_output.cpu().numpy())
    val_loss /= len(val_dataloader)
    val_auroc = roc_auc_score(all_labels, all_outputs)
    print(f'Validation Loss: {val_loss:.4f}, Validation AUROC: {val_auroc:.4f}')


    early_stopping(val_loss, model, epoch)
    if early_stopping.early_stop:
        print(f'Early stopping at epoch {epoch+1}')
        break


NameError: name 'num_epochs' is not defined

In [125]:
ig_scores_after_training = torch.sigmoid(model.immunogenicity.ig)
ig_score = {
    'Gene': all_genes,
    'IG Score Before Training': [score.item() for score in ig_scores_before_training],
    'IG Score After Training': [score.item() for score in ig_scores_after_training]
}  
df = pd.DataFrame(ig_score)


In [123]:

# Calculate the difference and add it as a new column
df['Difference'] = df['IG Score After Training'] - df['IG Score Before Training']


/tmp/ipykernel_216404/3660844667.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Difference'] = df['IG Score After Training'] - df['IG Score Before Training']


In [112]:
df

,Gene,IG Score Before Training,IG Score After Training,Difference
0,AFP,0.268941,0.268941,0.0
1,CEACAM21,0.268941,0.268941,0.0
2,CEACAM4,0.268941,0.268941,0.0
3,CEACAM7,0.268941,0.268941,0.0
4,CEACAM5,0.268941,0.268941,0.0
...,...,...,...,...
263,KRTAP12-4,0.268941,0.268941,0.0
264,KRTAP12-3,0.268941,0.268941,0.0
265,KRTAP12-2,0.268941,0.268941,0.0
266,KRTAP12-1,0.268941,0.268941,0.0


In [127]:

# Sort the DataFrame by the Difference column in descending order
df = df.sort_values(by='IG Score After Training', ascending=False)


In [114]:
df['rank'] = df['Difference'].rank(ascending=False)

In [128]:
df.head(10)

,Gene,IG Score Before Training,IG Score After Training
13,MUC1,0.268941,0.273784
94,MAGED1,0.268941,0.273364
155,KRT18,0.268941,0.269840
190,KRTAP4-1,0.268941,0.268941
189,KRTAP4-2,0.268941,0.268941
188,KRTAP4-3,0.268941,0.268941
203,KRT33B,0.268941,0.268941
202,KRT33A,0.268941,0.268941
201,KRTAP17-1,0.268941,0.268941
145,KRT73,0.268941,0.268941


In [116]:
import os
# Write the sorted DataFrame to a CSV file
df.to_csv('./changes.csv', index=False)

torch.save(model.state_dict(), './final_model.pth')

In [117]:
df =  df[df['Difference'] != 0]
df

,Gene,IG Score Before Training,IG Score After Training,Difference,rank
13,MUC1,0.268941,0.273784,0.004843,1.0
94,MAGED1,0.268941,0.273364,0.004423,2.0
155,KRT18,0.268941,0.269840,0.000899,3.0
217,KRT17,0.268941,0.267564,-0.001377,257.0
141,KRT5,0.268941,0.266354,-0.002587,258.0
75,PMEL,0.268941,0.259548,-0.009393,259.0
97,MAGED2,0.268941,0.258832,-0.010109,260.0
116,KRTCAP2,0.268941,0.258829,-0.010112,261.0
24,EPCAM,0.268941,0.257976,-0.010965,262.0
22,MLANA,0.268941,0.257363,-0.011579,263.0


In [118]:
tumor_antigen = pd.read_csv('tumor_antigen.csv')

In [119]:
common_genes = pd.merge(df, tumor_antigen[['Gene']], on='Gene')

In [120]:
common_genes

,Gene,IG Score Before Training,IG Score After Training,Difference,rank
0,MUC1,0.268941,0.273784,0.004843,1.0
1,MAGED1,0.268941,0.273364,0.004423,2.0
2,KRT18,0.268941,0.269840,0.000899,3.0
3,KRT17,0.268941,0.267564,-0.001377,257.0
4,KRT5,0.268941,0.266354,-0.002587,258.0
5,PMEL,0.268941,0.259548,-0.009393,259.0
6,MAGED2,0.268941,0.258832,-0.010109,260.0
7,KRTCAP2,0.268941,0.258829,-0.010112,261.0
8,EPCAM,0.268941,0.257976,-0.010965,262.0
9,MLANA,0.268941,0.257363,-0.011579,263.0
